## **Authors**
#### **Name(s):** Ibrahim Labib, Shahd Kazan, and Islam Abdelmotaleb
#### **ID(s):** 232400591, 232400237, and 232400623
---

# **Advanced Deep Learning**
# **Final Project:** *Build a Smart Q&A Assistant with Hybrid Search*
---

## **Project Overview**
**Description:** This section introduces the purpose of the project: building a Smart Q&A Assistant that uses hybrid search to answer technical AI and Data Science questions. The notebook combines keyword retrieval, semantic retrieval, hybrid fusion, retrieval evaluation, RAG-style answer generation, citation handling, and an interactive Gradio interface. It also explains the overall workflow so the reader understands how the system moves from raw documents to a live question-answering demo.

In [98]:
# Added comment: This first code cell is intentionally only descriptive comments.
# Added comment: The real executable pipeline starts in the following configuration/import cells.

# ============================================================
# Smart Q&A Assistant with Hybrid Search
# Final Project Notebook
# ============================================================
#
# What this notebook builds:
# 1. Loads the StackLite-style JSON corpus.
# 2. Cleans HTML question text.
# 3. Splits documents into searchable passages.
# 4. Builds BM25 keyword retrieval.
# 5. Builds MiniLM semantic embedding retrieval.
# 6. Combines BM25 and semantic retrieval using hybrid fusion.
# 7. Evaluates retrieval using MAP@10, MRR@10, and nDCG@10.
# 8. Generates RAG answers with citations from retrieved sources.
# 9. Launches a Gradio interface for live Q&A testing.


## **Setup 1: *Project Configuration***
**Description:** This section defines the main project settings used throughout the notebook. It stores the project name, dataset file names, embedding model name, local generation model name, retrieval settings, chunking parameters, RAG context length, and demo options. Keeping these values in one place makes the notebook easier to modify and reuse.

In [99]:
# ============================================================
# Project configuration
# ============================================================
# Added comment: This configuration cell centralizes values used by later cells.
# Added comment: If the dataset names, model name, retrieval depth, or chunk size changes, update them here first.


# Added comment: Project-level name used in print statements and the Gradio app title.
PROJECT_NAME = "Smart Q&A Assistant with Hybrid Search"

# Names of the two dataset files.
# In Colab, upload these exact files if they are not already present.
# Added comment: These are the required input files. Keep the names aligned with the uploaded JSON files.
DATASET_FILENAMES = [
    "top_ai_questions.json",
    "top_datascience_questions.json",
]

# Embedding model for semantic search.
# MiniLM is small, fast, and suitable for free Colab GPUs/CPUs.
# Added comment: MiniLM is selected because it is accurate enough for semantic search and lightweight enough for Colab.
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

# Local open-source model used if no OpenAI API key is available.
# flan-t5-base gives better answers than flan-t5-small but is still manageable in Colab.
LOCAL_LLM_NAME = "google/flan-t5-base"

# Optional OpenAI model.
# The notebook will only use it if OPENAI_API_KEY exists in the environment.
OPENAI_MODEL = "gpt-4o-mini"

# Chunking settings.
# Each StackExchange question can be long, so we split long documents into passages.
CHUNK_WORDS = 180
CHUNK_OVERLAP = 40

# Retrieval settings.
# Added comment: Retrieval settings control how many sources are shown and evaluated.
DEFAULT_TOP_K = 5
EVALUATION_TOP_K = 10
RRF_K = 60

# Generator settings.
MAX_CONTEXT_CHARS = 7000
MAX_NEW_TOKENS = 350

# Set this to False if you want the notebook to build the system without running
# the sample generated answer automatically.
RUN_DEMO_RAG_ANSWER = True

# Set this to False if you already installed all libraries.
# Added comment: Set this to False only if every dependency is already installed in the runtime.
INSTALL_PACKAGES = True


## **Setup 2: *Install and Import Libraries***
**Description:** This section installs and imports all required Python libraries. The notebook uses packages such as `rank_bm25` for BM25 retrieval, `sentence-transformers` for MiniLM embeddings, `BeautifulSoup` for HTML cleaning, `transformers` for model loading, and `gradio` for the user interface. It also checks whether a GPU is available and prints basic runtime information.

In [100]:
# ============================================================
# Install and import libraries
# ============================================================
# Added comment: This cell installs missing dependencies and imports the libraries used across the notebook.
# Added comment: The helper function avoids reinstalling packages that already exist in the Colab environment.


import os
import re
import sys
import json
import math
import time
import html
import glob
import random
import shutil
import inspect
import subprocess
import importlib
from collections import defaultdict, Counter
from typing import Dict, List, Tuple, Any, Set

# Added comment: This helper keeps the notebook easy to run on fresh Colab sessions.
def install_if_missing(pip_name: str, import_name: str = None) -> None:
    """
    Install a package only if it is missing.

    pip_name:
        Name used by pip install.
    import_name:
        Name used by import. If not supplied, pip_name is used.
    """
    if import_name is None:
        import_name = pip_name

    try:
        importlib.import_module(import_name)
    except ImportError:
        print(f"Installing missing package: {pip_name}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])

# Added comment: The notebook installs only the packages needed for retrieval, generation, and UI.
if INSTALL_PACKAGES:
    install_if_missing("rank_bm25", "rank_bm25")
    install_if_missing("sentence-transformers", "sentence_transformers")
    install_if_missing("beautifulsoup4", "bs4")
    install_if_missing("gradio", "gradio")
    install_if_missing("transformers", "transformers")
    install_if_missing("accelerate", "accelerate")
    install_if_missing("openai", "openai")

import numpy as np
import pandas as pd
from bs4 import BeautifulSoup
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
import torch
from transformers import pipeline
import gradio as gr
from IPython.display import display, HTML

try:
    from IPython.display import display, Markdown
except Exception:
    display = print
    Markdown = lambda x: x

# Added comment: Seeds make any random behavior more reproducible across runs.
random.seed(42)
np.random.seed(42)

# Added comment: These print statements confirm the environment before the main pipeline starts.
print(f"Project: {PROJECT_NAME}")
print(f"Python version: {sys.version.split()[0]}")
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU name: {torch.cuda.get_device_name(0)}")


Project: Smart Q&A Assistant with Hybrid Search
Python version: 3.12.13
GPU available: True
GPU name: NVIDIA A100-SXM4-40GB


## **Setup 3: *Dataset Loading***
**Description:** This section locates and loads the two JSON dataset files: `top_ai_questions.json` and `top_datascience_questions.json`. The code searches common Colab paths, asks the user to upload missing files if needed, and loads the JSON records into Python lists. It then prints the raw dataset sizes so the user can confirm that both datasets were loaded correctly.

In [101]:
# ============================================================
# Locate and load the dataset files
# ============================================================
# Added comment: This cell makes the notebook portable by searching common Colab paths before asking for upload.
# Added comment: The output confirms whether both dataset JSON files were found and loaded correctly.


def find_dataset_file(filename: str) -> str:
    """
    Search common Colab/local locations for a dataset file.

    This makes the notebook easier to run because Colab paths may differ.
    """
    # Added comment: Check the current folder, /content, and /mnt/data because Colab and ChatGPT use different paths.
    candidate_paths = [
        os.path.join(os.getcwd(), filename),
        os.path.join("/content", filename),
        os.path.join("/mnt/data", filename),
    ]

    # Added comment: Return the first matching path so later code can open the file safely.
    for path in candidate_paths:
        if os.path.exists(path):
            return path

    # Search one level below /content. This catches common upload/folder cases.
    for path in glob.glob(f"/content/**/{filename}", recursive=True):
        if os.path.exists(path):
            return path

    return ""

# Added comment: This upload helper is only triggered if one or more dataset files were not found automatically.
def upload_missing_files(missing_files: List[str]) -> None:
    """
    Ask the user to upload missing files when running in Google Colab.

    If this is not Colab, it prints instructions instead of crashing silently.
    """
    if not missing_files:
        return

    print("The following dataset files were not found:")
    for name in missing_files:
        print(f"- {name}")

    try:
        from google.colab import files
        print("\nPlease upload the missing JSON files now.")
        files.upload()
    except Exception:
        print("\nThis does not appear to be Google Colab.")
        print("Place the missing JSON files in the same folder as this notebook and run again.")

# First search for all required files.
# Added comment: Store discovered dataset paths and track missing files before loading JSON.
dataset_paths = {}
missing = []

for filename in DATASET_FILENAMES:
    path = find_dataset_file(filename)
    if path:
        dataset_paths[filename] = path
    else:
        missing.append(filename)

# If files are missing, ask for upload, then search again.
if missing:
    upload_missing_files(missing)

    dataset_paths = {}
    missing = []

    for filename in DATASET_FILENAMES:
        path = find_dataset_file(filename)
        if path:
            dataset_paths[filename] = path
        else:
            missing.append(filename)

if missing:
    raise FileNotFoundError(
        "Dataset files are still missing. Please upload: " + ", ".join(missing)
    )

print("\nDataset files found:")
for filename, path in dataset_paths.items():
    print(f"- {filename}: {path}")

def load_json_list(path: str) -> List[Dict[str, Any]]:
    """
    Load a JSON file expected to contain a list of StackExchange-like questions.
    """
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    if not isinstance(data, list):
        raise ValueError(f"{path} must contain a JSON list.")

    return data

# Added comment: Load both JSON lists into memory so preprocessing can convert them into documents.
raw_ai_questions = load_json_list(dataset_paths["top_ai_questions.json"])
raw_ds_questions = load_json_list(dataset_paths["top_datascience_questions.json"])

print("\nRaw dataset sizes:")
print(f"- AI questions: {len(raw_ai_questions)}")
print(f"- Data Science questions: {len(raw_ds_questions)}")
print(f"- Total raw questions: {len(raw_ai_questions) + len(raw_ds_questions)}")



Dataset files found:
- top_ai_questions.json: /content/top_ai_questions.json
- top_datascience_questions.json: /content/top_datascience_questions.json

Raw dataset sizes:
- AI questions: 750
- Data Science questions: 750
- Total raw questions: 1500


## **Setup 4: *Data Preprocessing***
**Description:** This section cleans and standardizes the raw StackExchange-style records. It removes HTML tags from question bodies, decodes special characters, cleans titles, extracts metadata such as tags and links, and builds a unified document structure. Each final document contains a title, tags, body text, source name, URL, and combined searchable text.

In [102]:
# ============================================================
# Clean and normalize the corpus
# ============================================================
# Added comment: This cell converts raw StackExchange-style JSON records into clean searchable documents.
# Added comment: Cleaning is important because retrieval models perform better on plain text than raw HTML.


# Added comment: HTML cleaning is the first step because StackExchange bodies are stored as HTML.
def clean_html(raw_html: str) -> str:
    """
    Convert HTML from StackExchange posts into readable plain text.

    StackExchange bodies contain tags such as <p>, <code>, <blockquote>, etc.
    Search models work better on clean text than raw HTML.
    """
    if raw_html is None:
        return ""

    # Added comment: BeautifulSoup parses HTML safely and lets us extract visible text.
    soup = BeautifulSoup(raw_html, "html.parser")

    # Make code blocks readable as text.
    for code_tag in soup.find_all(["code", "pre"]):
        code_text = code_tag.get_text(" ", strip=True)
        code_tag.replace_with(f" CODE_SNIPPET {code_text} ")

    # Extract visible text.
    text = soup.get_text(" ", strip=True)

    # Decode entities such as &quot; and &#39;.
    text = html.unescape(text)

    # Normalize whitespace.
    text = re.sub(r"\s+", " ", text).strip()

    return text

def clean_title(title: str) -> str:
    """
    Clean title text.
    """
    if title is None:
        return ""
    title = html.unescape(str(title))
    title = re.sub(r"\s+", " ", title).strip()
    return title

def normalize_for_matching(text: str) -> str:
    """
    Normalize text for simple title matching in evaluation.
    """
    text = html.unescape(str(text)).lower()
    text = re.sub(r"[^a-z0-9]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

# Added comment: This function creates one consistent document dictionary per StackExchange question.
def build_documents(raw_items: List[Dict[str, Any]], source_name: str) -> List[Dict[str, Any]]:
    """
    Convert raw JSON records into a clean document list.

    Each document represents one StackExchange question.
    """
    documents = []

    # Added comment: Each item is one raw question record from the JSON dataset.
    for item in raw_items:
        question_id = item.get("question_id", "unknown")
        title = clean_title(item.get("title", ""))
        body = clean_html(item.get("body", ""))
        tags = item.get("tags", [])
        link = item.get("link", "")
        score = item.get("score", 0)
        view_count = item.get("view_count", 0)
        answer_count = item.get("answer_count", 0)
        is_answered = item.get("is_answered", None)

        if not title and not body:
            continue

        tag_text = ", ".join(tags) if isinstance(tags, list) else str(tags)

        # Full text used for retrieval.
        # We keep the title and tags because they are strong signals for search.
        full_text = (
            f"Title: {title}\n"
            f"Tags: {tag_text}\n"
            f"Question: {body}"
        ).strip()

        documents.append({
            "doc_id": f"{source_name}_{question_id}",
            "question_id": question_id,
            "source": source_name,
            "title": title,
            "body": body,
            "tags": tags,
            "tag_text": tag_text,
            "link": link,
            "score": score,
            "view_count": view_count,
            "answer_count": answer_count,
            "is_answered": is_answered,
            "full_text": full_text,
        })

    return documents

# Added comment: Build separate document lists first so the source platform remains visible in results.
ai_documents = build_documents(raw_ai_questions, "AI StackExchange")
ds_documents = build_documents(raw_ds_questions, "Data Science StackExchange")

documents = ai_documents + ds_documents

print("\nClean document counts:")
print(f"- AI documents: {len(ai_documents)}")
print(f"- Data Science documents: {len(ds_documents)}")
print(f"- Total documents: {len(documents)}")

# Show a small preview so the user can confirm the data loaded correctly.
preview_df = pd.DataFrame([
    {
        "doc_id": d["doc_id"],
        "source": d["source"],
        "title": d["title"],
        "tags": d["tag_text"],
        "link": d["link"],
    }
    for d in documents[:5]
])
display(preview_df)



Clean document counts:
- AI documents: 750
- Data Science documents: 750
- Total documents: 1500


,doc_id,source,title,tags,link
0,AI StackExchange_1768,AI StackExchange,Could a paradox kill an AI?,"philosophy, decision-theory, mythology-of-ai, ...",https://ai.stackexchange.com/questions/1768/co...
1,AI StackExchange_35,AI StackExchange,What is the difference between artificial inte...,"machine-learning, comparison, terminology, ai-...",https://ai.stackexchange.com/questions/35/what...
2,AI StackExchange_111,AI StackExchange,How could self-driving cars make ethical decis...,"philosophy, ethics, autonomous-vehicles, decis...",https://ai.stackexchange.com/questions/111/how...
3,AI StackExchange_2008,AI StackExchange,How can neural networks deal with varying inpu...,"neural-networks, machine-learning, deep-learni...",https://ai.stackexchange.com/questions/2008/ho...
4,AI StackExchange_1479,AI StackExchange,Do scientists know what is happening inside ar...,"neural-networks, deep-learning, convolutional-...",https://ai.stackexchange.com/questions/1479/do...


## **Setup 5: *Passage Chunking***
**Description:** This section splits long documents into smaller searchable passages. The code uses word-based chunking with overlap so that long question posts are divided into focused chunks while still preserving context between chunks. These passages become the basic search units used by BM25, semantic retrieval, hybrid retrieval, and RAG generation.

In [103]:
# ============================================================
# Split documents into passages/chunks
# ============================================================
# Added comment: This cell creates smaller overlapping passages from full documents.
# Added comment: These chunks are what BM25, semantic search, hybrid retrieval, and RAG generation search over.


def word_chunk_text(text: str, chunk_words: int = 180, overlap: int = 40) -> List[str]:
    """
    Split long text into overlapping word chunks.

    Why chunking matters:
    - BM25 can search whole documents, but RAG works better with smaller passages.
    - Semantic embeddings also perform better when each chunk is focused.
    """
    # Added comment: Splitting by words keeps chunking simple and easy to understand.
    words = text.split()

    if len(words) <= chunk_words:
        return [" ".join(words)]

    chunks = []
    start = 0

    # Added comment: Move through the document in overlapping windows until all words are covered.
    while start < len(words):
        end = min(start + chunk_words, len(words))
        chunk = " ".join(words[start:end])
        chunks.append(chunk)

        # Stop if we reached the end.
        if end == len(words):
            break

        # Move forward while keeping overlap with the previous chunk.
        start = end - overlap

        # Safety check to avoid infinite loops.
        if start < 0:
            start = 0

    return chunks

def build_chunks(documents: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    """
    Build searchable chunks from documents.

    Each chunk keeps the original document metadata so we can cite sources.
    """
    chunks = []

    for doc in documents:
        text_chunks = word_chunk_text(
            doc["full_text"],
            chunk_words=CHUNK_WORDS,
            overlap=CHUNK_OVERLAP,
        )

        for chunk_number, chunk_text in enumerate(text_chunks):
            chunks.append({
                "chunk_id": f"{doc['doc_id']}_chunk_{chunk_number}",
                "chunk_number": chunk_number,
                "doc_id": doc["doc_id"],
                "source": doc["source"],
                "title": doc["title"],
                "tags": doc["tags"],
                "tag_text": doc["tag_text"],
                "link": doc["link"],
                "score": doc["score"],
                "view_count": doc["view_count"],
                "answer_count": doc["answer_count"],
                "text": chunk_text,
            })

    return chunks

# Added comment: This is the final searchable passage collection used by all retrieval methods.
chunks = build_chunks(documents)

print("\nChunking summary:")
print(f"- Total documents: {len(documents)}")
print(f"- Total chunks/passages: {len(chunks)}")
print(f"- Average chunks per document: {len(chunks) / max(len(documents), 1):.2f}")

chunk_preview_df = pd.DataFrame([
    {
        "chunk_id": c["chunk_id"],
        "source": c["source"],
        "title": c["title"],
        "text_preview": c["text"][:180] + "...",
    }
    for c in chunks[:5]
])
display(chunk_preview_df)



Chunking summary:
- Total documents: 1500
- Total chunks/passages: 2026
- Average chunks per document: 1.35


,chunk_id,source,title,text_preview
0,AI StackExchange_1768_chunk_0,AI StackExchange,Could a paradox kill an AI?,Title: Could a paradox kill an AI? Tags: philo...
1,AI StackExchange_35_chunk_0,AI StackExchange,What is the difference between artificial inte...,Title: What is the difference between artifici...
2,AI StackExchange_111_chunk_0,AI StackExchange,How could self-driving cars make ethical decis...,Title: How could self-driving cars make ethica...
3,AI StackExchange_111_chunk_1,AI StackExchange,How could self-driving cars make ethical decis...,the difference between a human being and an an...
4,AI StackExchange_2008_chunk_0,AI StackExchange,How can neural networks deal with varying inpu...,Title: How can neural networks deal with varyi...


## **Milestone 1: *Build BM25 Retrieval (Part 1)***
**Description:** This section builds the BM25 keyword-based retrieval system. It tokenizes the cleaned passage text, removes common stopwords, keeps useful technical tokens, and creates a BM25 index using the `rank_bm25` library. BM25 is used as the lexical search baseline because it works well when the user query contains exact technical terms found in the corpus.

In [104]:
# ============================================================
# Build BM25 lexical retrieval
# ============================================================
# Added comment: This cell builds the lexical keyword index using BM25.
# Added comment: BM25 is strong for exact technical terms such as fit_transform, LSTM, class_weights, and cross entropy.


# Added comment: Stopwords are common words that usually add little value to keyword retrieval.
STOPWORDS = {
    "a", "an", "and", "are", "as", "at", "be", "been", "but", "by",
    "can", "could", "do", "does", "for", "from", "had", "has", "have",
    "how", "i", "if", "in", "into", "is", "it", "its", "me", "my",
    "not", "of", "on", "or", "our", "should", "so", "that", "the",
    "their", "there", "these", "this", "to", "use", "used", "using",
    "was", "we", "what", "when", "where", "which", "who", "why",
    "will", "with", "would", "you", "your"
}

def tokenize(text: str) -> List[str]:
    """
    Tokenize text for BM25.

    BM25 is lexical search:
    - It matches exact words.
    - It is strong when the query and document use similar terms.
    """
    text = html.unescape(str(text)).lower()

    # Keep technical tokens like gpt-4, c++, c#, tf-idf, and numbers.
    # Added comment: The regex keeps technical tokens that normal word tokenizers might split badly.
    tokens = re.findall(r"[a-z0-9_+#.\-]+", text)

    # Remove stopwords and very short non-technical tokens.
    filtered = []
    for token in tokens:
        token = token.strip(".-")
        if not token:
            continue
        if token in STOPWORDS:
            continue
        if len(token) == 1 and token not in {"c", "r"}:
            continue
        filtered.append(token)

    return filtered

print("\nBuilding BM25 index...")
# Added comment: BM25 needs tokenized passages, so every chunk is converted into a token list.
tokenized_chunks = [tokenize(c["text"]) for c in chunks]
bm25_index = BM25Okapi(tokenized_chunks)
print("BM25 index ready.")



Building BM25 index...
BM25 index ready.


## **Milestone 3: *Add Semantic Search (Part 1)***
**Description:** This section builds the semantic search component using the MiniLM sentence-transformer model. Each passage is converted into a dense embedding vector, allowing the system to retrieve passages based on meaning rather than exact word overlap. The resulting embedding matrix is used later to compare user questions against all indexed passages using cosine similarity.

In [105]:
# ============================================================
# Build semantic embedding retrieval with MiniLM
# ============================================================
# Added comment: This cell builds the dense semantic index using MiniLM embeddings.
# Added comment: Semantic embeddings allow the system to retrieve passages based on meaning, not just exact words.


print("\nLoading embedding model...")
# Added comment: Use GPU when available because embedding the corpus can be slow on CPU.
embedding_device = "cuda" if torch.cuda.is_available() else "cpu"
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME, device=embedding_device)
print(f"Embedding model loaded on: {embedding_device}")

print("\nEncoding corpus chunks. This can take a few minutes the first time...")
# Added comment: Include title and tags with the passage text because they improve semantic retrieval quality.
chunk_texts_for_embedding = [
    f"{c['title']}\nTags: {c['tag_text']}\n{c['text']}"
    for c in chunks
]

# Added comment: normalize_embeddings=True makes dot product equivalent to cosine similarity later.
chunk_embeddings = embedding_model.encode(
    chunk_texts_for_embedding,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)

chunk_embeddings = chunk_embeddings.astype("float32")

print(f"Semantic embedding matrix shape: {chunk_embeddings.shape}")
print("Semantic index ready.")



Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model loaded on: cuda

Encoding corpus chunks. This can take a few minutes the first time...


Batches:   0%|          | 0/32 [00:00<?, ?it/s]

Semantic embedding matrix shape: (2026, 384)
Semantic index ready.


## **Milestone 3: *Hybrid Retrieval and Fusion (Part 2)***
**Description:** This section defines the main retrieval functions for BM25 search, semantic search, and hybrid search. It includes helper functions for ranking results, formatting retrieved passages, Reciprocal Rank Fusion, and normalized score-based fusion. The hybrid methods combine keyword matching and semantic similarity so the system can retrieve stronger results across different question styles.

In [106]:
# ============================================================
# Retrieval helper functions
# ============================================================
# Added comment: This cell contains the main retrieval logic used by the rest of the system.
# Added comment: It supports BM25, semantic retrieval, Reciprocal Rank Fusion, and normalized score fusion.


# Added comment: The functions below are shared by BM25, semantic search, and hybrid search.
def top_indices_from_scores(scores: np.ndarray, top_k: int) -> List[int]:
    """
    Return indices of the top_k highest scores.
    """
    if len(scores) == 0:
        return []

    top_k = min(top_k, len(scores))

    # argsort is clear and fine for this corpus size.
    return np.argsort(scores)[::-1][:top_k].tolist()

def make_result(chunk_index: int, score_value: float, rank: int, method: str) -> Dict[str, Any]:
    """
    Convert a chunk index and score into a readable retrieval result.
    """
    c = chunks[chunk_index]

    return {
        "rank": rank,
        "method": method,
        "chunk_index": chunk_index,
        "chunk_id": c["chunk_id"],
        "doc_id": c["doc_id"],
        "source": c["source"],
        "title": c["title"],
        "tags": c["tag_text"],
        "link": c["link"],
        "score": float(score_value),
        "text": c["text"],
    }

def bm25_retrieve(query: str, top_k: int = 10) -> List[Dict[str, Any]]:
    """
    Retrieve passages using BM25 keyword search.
    """
    # Added comment: Convert the user query into the same token format used for BM25 indexing.
    query_tokens = tokenize(query)
    scores = np.array(bm25_index.get_scores(query_tokens), dtype="float32")

    top_indices = top_indices_from_scores(scores, top_k)

    results = []
    for rank, idx in enumerate(top_indices, start=1):
        results.append(make_result(idx, scores[idx], rank, "bm25"))

    return results

def semantic_retrieve(query: str, top_k: int = 10) -> List[Dict[str, Any]]:
    """
    Retrieve passages using semantic similarity.

    The query is embedded with MiniLM and compared against all chunk embeddings
    using cosine similarity. Because embeddings are normalized, dot product equals cosine similarity.
    """
    # Added comment: Convert the user query into a dense vector for semantic similarity search.
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True,
    ).astype("float32")[0]

    # Added comment: Because vectors are normalized, this dot product is cosine similarity.
    scores = chunk_embeddings @ query_embedding

    top_indices = top_indices_from_scores(scores, top_k)

    results = []
    for rank, idx in enumerate(top_indices, start=1):
        results.append(make_result(idx, scores[idx], rank, "semantic"))

    return results

def reciprocal_rank_fusion(
    result_lists: List[List[Dict[str, Any]]],
    top_k: int = 10,
    rrf_k: int = 60,
) -> List[Dict[str, Any]]:
    """
    Combine rankings using Reciprocal Rank Fusion.

    RRF idea:
    - A document gets points based on how high it appears in each ranking.
    - Score contribution = 1 / (rrf_k + rank)
    - If a passage appears high in both BM25 and semantic search, it rises.
    """
    # Added comment: RRF accumulates rank-based points from multiple retrieval result lists.
    fusion_scores = defaultdict(float)
    best_result_by_index = {}

    for results in result_lists:
        for result in results:
            idx = result["chunk_index"]
            rank = result["rank"]
            # Added comment: Higher-ranked results receive larger contributions to the fused score.
            fusion_scores[idx] += 1.0 / (rrf_k + rank)

            # Keep a copy of metadata for the result.
            if idx not in best_result_by_index:
                best_result_by_index[idx] = result

    ranked_indices = sorted(
        fusion_scores.keys(),
        key=lambda idx: fusion_scores[idx],
        reverse=True,
    )[:top_k]

    fused_results = []
    for rank, idx in enumerate(ranked_indices, start=1):
        result = dict(best_result_by_index[idx])
        result["rank"] = rank
        result["method"] = "hybrid_rrf"
        result["score"] = float(fusion_scores[idx])
        fused_results.append(result)

    return fused_results

def minmax_normalize(score_dict: Dict[int, float]) -> Dict[int, float]:
    """
    Min-max normalize scores to the range [0, 1].

    This is useful for score-based fusion because BM25 and cosine similarity
    are on different scales.
    """
    if not score_dict:
        return {}

    values = list(score_dict.values())
    min_value = min(values)
    max_value = max(values)

    if math.isclose(max_value, min_value):
        return {k: 1.0 for k in score_dict}

    return {
        k: (v - min_value) / (max_value - min_value)
        for k, v in score_dict.items()
    }

def hybrid_score_fusion(query: str, top_k: int = 10, bm25_weight: float = 0.50) -> List[Dict[str, Any]]:
    """
    Combine BM25 and semantic scores using normalized weighted scoring.

    hybrid_score = bm25_weight * normalized_bm25 + semantic_weight * normalized_semantic
    """
    semantic_weight = 1.0 - bm25_weight

    # Retrieve more candidates from each method before fusing.
    # Added comment: Retrieve extra candidates first, then fuse and keep only the best top_k.
    candidate_k = min(max(top_k * 5, 50), len(chunks))

    bm25_results = bm25_retrieve(query, top_k=candidate_k)
    semantic_results = semantic_retrieve(query, top_k=candidate_k)

    bm25_scores = {r["chunk_index"]: r["score"] for r in bm25_results}
    semantic_scores = {r["chunk_index"]: r["score"] for r in semantic_results}

    candidate_indices = set(bm25_scores.keys()) | set(semantic_scores.keys())

    bm25_norm = minmax_normalize({idx: bm25_scores.get(idx, 0.0) for idx in candidate_indices})
    semantic_norm = minmax_normalize({idx: semantic_scores.get(idx, 0.0) for idx in candidate_indices})

    fused = {}
    for idx in candidate_indices:
        fused[idx] = bm25_weight * bm25_norm.get(idx, 0.0) + semantic_weight * semantic_norm.get(idx, 0.0)

    ranked_indices = sorted(fused.keys(), key=lambda idx: fused[idx], reverse=True)[:top_k]

    results = []
    for rank, idx in enumerate(ranked_indices, start=1):
        results.append(make_result(idx, fused[idx], rank, "hybrid_score"))

    return results

def retrieve(query: str, top_k: int = 10, mode: str = "hybrid_rrf") -> List[Dict[str, Any]]:
    """
    Main retrieval function.

    mode options:
    - "bm25": keyword search only
    - "semantic": embedding search only
    - "hybrid_rrf": BM25 + semantic using Reciprocal Rank Fusion
    - "hybrid_score": BM25 + semantic using normalized weighted scores
    """
    mode = mode.lower().strip()

    # Added comment: Route the query to the selected retrieval method.
    if mode == "bm25":
        return bm25_retrieve(query, top_k=top_k)

    if mode == "semantic":
        return semantic_retrieve(query, top_k=top_k)

    if mode == "hybrid_rrf":
        bm25_results = bm25_retrieve(query, top_k=min(50, len(chunks)))
        semantic_results = semantic_retrieve(query, top_k=min(50, len(chunks)))
        return reciprocal_rank_fusion([bm25_results, semantic_results], top_k=top_k, rrf_k=RRF_K)

    if mode == "hybrid_score":
        return hybrid_score_fusion(query, top_k=top_k, bm25_weight=0.50)

    raise ValueError("Unknown retrieval mode. Use: bm25, semantic, hybrid_rrf, or hybrid_score.")

def results_to_dataframe(results: List[Dict[str, Any]]) -> pd.DataFrame:
    """
    Convert retrieval results to a readable table.
    """
    rows = []

    for r in results:
        rows.append({
            "rank": r["rank"],
            "method": r["method"],
            "score": round(r["score"], 5),
            "source": r["source"],
            "title": r["title"],
            "tags": r["tags"],
            "link": r["link"],
            "text_preview": r["text"][:250] + ("..." if len(r["text"]) > 250 else ""),
        })

    return pd.DataFrame(rows)


## **Milestone 1: *BM25 Top-10 Results (Part 2)***
**Description:** This section demonstrates the BM25 retriever by running sample technical questions through the BM25 search function. For each sample question, the notebook displays the top 10 retrieved passages in a table with rank, score, source, title, tags, link, and text preview. This satisfies the first milestone by showing that keyword-based retrieval works on the indexed corpus.

In [107]:
# ============================================================
# Show BM25 top-10 results for sample questions
# ============================================================
# Added comment: This cell demonstrates Milestone 1 by showing BM25 top-10 results for sample questions.
# Added comment: The displayed tables make it easy to inspect whether lexical retrieval is working correctly.


# Added comment: These sample questions are used to visually verify BM25 retrieval behavior.
sample_questions = [
    "What is the difference between artificial intelligence and machine learning?",
    "How can neural networks handle varying input sizes?",
    "What is the difference between fit and fit_transform in scikit-learn?",
]

print("\nBM25 top-10 results for sample questions:")
for question in sample_questions:
    print("\n" + "=" * 90)
    print(f"Question: {question}")
    # Added comment: Displaying ranked results helps confirm whether the correct topics are retrieved.
    bm25_results = bm25_retrieve(question, top_k=10)
    display(results_to_dataframe(bm25_results))



BM25 top-10 results for sample questions:

Question: What is the difference between artificial intelligence and machine learning?


,rank,method,score,source,title,tags,link,text_preview
0,1,bm25,23.60237,Data Science StackExchange,Difference between machine learning and artifi...,"machine-learning, ai, theory",https://datascience.stackexchange.com/question...,Title: Difference between machine learning and...
1,2,bm25,18.37688,AI StackExchange,What is the difference between artificial inte...,"terminology, robotics, robots, comparison",https://ai.stackexchange.com/questions/1462/wh...,Title: What is the difference between artifici...
2,3,bm25,17.11793,AI StackExchange,What is the difference between artificial inte...,"terminology, comparison, cognitive-science",https://ai.stackexchange.com/questions/1847/wh...,Title: What is the difference between artifici...
3,4,bm25,16.89057,AI StackExchange,What is the difference between artificial inte...,"machine-learning, comparison, terminology, ai-...",https://ai.stackexchange.com/questions/35/what...,Title: What is the difference between artifici...
4,5,bm25,15.76859,AI StackExchange,What are the top artificial intelligence journ...,research,https://ai.stackexchange.com/questions/2306/wh...,Title: What are the top artificial intelligenc...
5,6,bm25,15.61955,AI StackExchange,What is the difference between artificial inte...,"comparison, terminology, definitions",https://ai.stackexchange.com/questions/7446/wh...,Title: What is the difference between artifici...
6,7,bm25,15.06335,AI StackExchange,How is Bayes' Theorem used in artificial intel...,"machine-learning, applications, bayes-theorem",https://ai.stackexchange.com/questions/2738/ho...,Title: How is Bayes' Theorem used in artificia...
7,8,bm25,14.37736,AI StackExchange,What are examples of techniques to prevent bia...,"machine-learning, datasets, social, ai-safety,...",https://ai.stackexchange.com/questions/3175/wh...,Title: What are examples of techniques to prev...
8,9,bm25,14.11729,AI StackExchange,What is the difference between machine learnin...,"machine-learning, deep-learning, comparison",https://ai.stackexchange.com/questions/1742/wh...,Title: What is the difference between machine ...
9,10,bm25,13.21772,AI StackExchange,"What is ""early stopping"" in machine learning?","deep-learning, definitions, overfitting, regul...",https://ai.stackexchange.com/questions/16/what...,"Title: What is ""early stopping"" in machine lea..."



Question: How can neural networks handle varying input sizes?


,rank,method,score,source,title,tags,link,text_preview
0,1,bm25,23.45489,AI StackExchange,How can neural networks deal with varying inpu...,"neural-networks, machine-learning, deep-learni...",https://ai.stackexchange.com/questions/2008/ho...,Title: How can neural networks deal with varyi...
1,2,bm25,16.61127,Data Science StackExchange,How to feed LSTM with different input array si...,"keras, lstm",https://datascience.stackexchange.com/question...,Title: How to feed LSTM with different input a...
2,3,bm25,13.47343,AI StackExchange,How do I handle large images when training a CNN?,"neural-networks, deep-learning, image-recognit...",https://ai.stackexchange.com/questions/3938/ho...,Title: How do I handle large images when train...
3,4,bm25,13.43449,AI StackExchange,How to handle rectangular images in convolutio...,"neural-networks, convolutional-neural-networks...",https://ai.stackexchange.com/questions/8323/ho...,Title: How to handle rectangular images in con...
4,5,bm25,12.19822,AI StackExchange,Are there neural networks with very few nodes ...,neural-networks,https://ai.stackexchange.com/questions/10944/a...,Title: Are there neural networks with very few...
5,6,bm25,10.87494,AI StackExchange,How can Transformers handle arbitrary length i...,"natural-language-processing, recurrent-neural-...",https://ai.stackexchange.com/questions/22957/h...,Title: How can Transformers handle arbitrary l...
6,7,bm25,9.24830,AI StackExchange,Can a single neural network handle recognizing...,"neural-networks, image-recognition",https://ai.stackexchange.com/questions/13/can-...,"need for rapid, real-time processing, the maxi..."
7,8,bm25,8.84010,Data Science StackExchange,How to prepare the varied size input in CNN pr...,"machine-learning, neural-network, deep-learnin...",https://datascience.stackexchange.com/question...,Title: How to prepare the varied size input in...
8,9,bm25,8.27890,AI StackExchange,Why do we need floats for using neural networks?,"neural-networks, machine-learning",https://ai.stackexchange.com/questions/7247/wh...,Title: Why do we need floats for using neural ...
9,10,bm25,8.17266,AI StackExchange,Is it possible to train the neural network to ...,"neural-networks, math, applications",https://ai.stackexchange.com/questions/154/is-...,Title: Is it possible to train the neural netw...



Question: What is the difference between fit and fit_transform in scikit-learn?


,rank,method,score,source,title,tags,link,text_preview
0,1,bm25,28.72890,Data Science StackExchange,What's the difference between fit and fit_tran...,"python, scikit-learn",https://datascience.stackexchange.com/question...,Title: What's the difference between fit and f...
1,2,bm25,15.18495,Data Science StackExchange,What is the difference between fit() and fit_g...,"deep-learning, keras",https://datascience.stackexchange.com/question...,Title: What is the difference between fit() an...
2,3,bm25,14.24369,Data Science StackExchange,Data normalization before or after train-test ...,normalization,https://datascience.stackexchange.com/question...,Title: Data normalization before or after trai...
3,4,bm25,12.08271,Data Science StackExchange,Difference between OrdinalEncoder and LabelEnc...,"machine-learning, python, scikit-learn, prepro...",https://datascience.stackexchange.com/question...,Title: Difference between OrdinalEncoder and L...
4,5,bm25,11.99171,AI StackExchange,What is the difference between the triplet los...,"comparison, objective-functions, siamese-neura...",https://ai.stackexchange.com/questions/35977/w...,Title: What is the difference between the trip...
5,6,bm25,10.95438,Data Science StackExchange,What's the difference between Sklearn F1 score...,"classification, scikit-learn, metric",https://datascience.stackexchange.com/question...,Title: What's the difference between Sklearn F...
6,7,bm25,10.84687,Data Science StackExchange,What is the difference between cross_validate ...,"machine-learning, scikit-learn",https://datascience.stackexchange.com/question...,Title: What is the difference between cross_va...
7,8,bm25,10.18193,Data Science StackExchange,What is the difference between CountVectorizer...,"machine-learning, nlp, scikit-learn",https://datascience.stackexchange.com/question...,Title: What is the difference between CountVec...
8,9,bm25,10.07310,Data Science StackExchange,How can I fit categorical data types for rando...,"python, scikit-learn, data-mining, random-forest",https://datascience.stackexchange.com/question...,Title: How can I fit categorical data types fo...
9,10,bm25,9.68391,Data Science StackExchange,When to use One Hot Encoding vs LabelEncoder v...,"scikit-learn, categorical-data, feature-engine...",https://datascience.stackexchange.com/question...,Title: When to use One Hot Encoding vs LabelEn...


## **Milestone 2: *Evaluate Retrieval***
**Description:** This section evaluates the retrieval system using manually selected evaluation questions with known relevant documents. It defines MAP@10, MRR@10, and nDCG@10, then compares BM25, semantic search, hybrid RRF, and hybrid score fusion. The output table shows how well each retrieval method ranks relevant documents in the top 10 results.

In [108]:
# ============================================================
# Evaluation set and retrieval metrics
# ============================================================
# Added comment: This cell evaluates the retrieval methods using MAP@10, MRR@10, and nDCG@10.
# Added comment: The evaluation is document-level, so repeated chunks from the same document are deduplicated.


# These evaluation queries are tied to known titles in the uploaded corpus.
# The relevant document is found automatically by matching the expected title text.
# Added comment: Each evaluation case pairs a user-style query with a known relevant title phrase.
EVAL_CASES = [
    {
        "query": "What is the difference between artificial intelligence and machine learning?",
        "title_contains": "difference between artificial intelligence and machine learning",
    },
    {
        "query": "How can neural networks deal with varying input sizes?",
        "title_contains": "neural networks deal with varying input sizes",
    },
    {
        "query": "Why do we need explainable AI?",
        "title_contains": "why do we need explainable ai",
    },
    {
        "query": "What is self-supervised learning in machine learning?",
        "title_contains": "what is self supervised learning in machine learning",
    },
    {
        "query": "What is the difference between model-free and model-based reinforcement learning?",
        "title_contains": "difference between model free and model based reinforcement learning",
    },
    {
        "query": "Why is Python such a popular language in AI?",
        "title_contains": "why is python such a popular language in the ai field",
    },
    {
        "query": "What is the difference between fit and fit_transform in scikit-learn?",
        "title_contains": "difference between fit and fit transform in scikit learn",
    },
    {
        "query": "How do I set class weights for imbalanced classes in Keras?",
        "title_contains": "how to set class weights for imbalanced classes in keras",
    },
    {
        "query": "What is micro average versus macro average performance?",
        "title_contains": "micro average vs macro average performance",
    },
    {
        "query": "What is positional encoding in the transformer model?",
        "title_contains": "what is the positional encoding in the transformer model",
    },
]

# Added comment: Relevance labels are created automatically by matching expected title text.
def find_relevant_doc_ids(title_contains: str) -> Set[str]:
    """
    Find document IDs whose title contains the target phrase.

    This creates a simple relevance set for evaluation.
    """
    target = normalize_for_matching(title_contains)
    relevant = set()

    for doc in documents:
        title_norm = normalize_for_matching(doc["title"])
        if target in title_norm:
            relevant.add(doc["doc_id"])

    return relevant

# Attach relevant document IDs to each evaluation case.
valid_eval_cases = []
for case in EVAL_CASES:
    relevant_doc_ids = find_relevant_doc_ids(case["title_contains"])
    case = dict(case)
    case["relevant_doc_ids"] = relevant_doc_ids

    if relevant_doc_ids:
        valid_eval_cases.append(case)
    else:
        print(f"Warning: no relevant document found for query: {case['query']}")

print(f"\nEvaluation queries with matched relevant documents: {len(valid_eval_cases)}")

def deduplicate_results_by_doc(results: List[Dict[str, Any]]) -> List[str]:
    """
    Convert chunk-level results to unique document-level predictions.

    If multiple chunks come from the same original question, we count the document once.
    """
    seen = set()
    doc_ids = []

    for r in results:
        doc_id = r["doc_id"]
        if doc_id not in seen:
            seen.add(doc_id)
            doc_ids.append(doc_id)

    return doc_ids

# Added comment: The next metric functions measure retrieval quality from different angles.
def average_precision_at_k(predicted_doc_ids: List[str], relevant_doc_ids: Set[str], k: int = 10) -> float:
    """
    Compute AP@k for one query.

    AP rewards systems that rank relevant documents earlier.
    """
    if not relevant_doc_ids:
        return 0.0

    predicted_doc_ids = predicted_doc_ids[:k]
    hits = 0
    precision_sum = 0.0

    for rank, doc_id in enumerate(predicted_doc_ids, start=1):
        if doc_id in relevant_doc_ids:
            hits += 1
            precision_sum += hits / rank

    return precision_sum / min(len(relevant_doc_ids), k)

def reciprocal_rank_at_k(predicted_doc_ids: List[str], relevant_doc_ids: Set[str], k: int = 10) -> float:
    """
    Compute RR@k for one query.

    RR is 1/rank of the first relevant result.
    """
    predicted_doc_ids = predicted_doc_ids[:k]

    for rank, doc_id in enumerate(predicted_doc_ids, start=1):
        if doc_id in relevant_doc_ids:
            return 1.0 / rank

    return 0.0

def dcg_at_k(predicted_doc_ids: List[str], relevant_doc_ids: Set[str], k: int = 10) -> float:
    """
    Compute DCG@k with binary relevance.

    Relevant documents receive gain 1, non-relevant documents receive gain 0.
    """
    score = 0.0

    for rank, doc_id in enumerate(predicted_doc_ids[:k], start=1):
        relevance = 1.0 if doc_id in relevant_doc_ids else 0.0
        score += relevance / math.log2(rank + 1)

    return score

def ndcg_at_k(predicted_doc_ids: List[str], relevant_doc_ids: Set[str], k: int = 10) -> float:
    """
    Compute nDCG@k.

    nDCG compares the actual ranking to the ideal ranking.
    """
    actual_dcg = dcg_at_k(predicted_doc_ids, relevant_doc_ids, k)

    ideal_length = min(len(relevant_doc_ids), k)
    ideal_doc_ids = list(relevant_doc_ids)[:ideal_length]
    ideal_dcg = dcg_at_k(ideal_doc_ids, relevant_doc_ids, k)

    if ideal_dcg == 0:
        return 0.0

    return actual_dcg / ideal_dcg

def evaluate_retriever(mode: str, top_k: int = 10) -> Dict[str, float]:
    """
    Evaluate one retrieval mode over all evaluation queries.
    """
    ap_scores = []
    rr_scores = []
    ndcg_scores = []

    for case in valid_eval_cases:
        results = retrieve(case["query"], top_k=top_k * 3, mode=mode)
        # Added comment: Multiple chunks can come from one document, so deduplicate before scoring.
        predicted_doc_ids = deduplicate_results_by_doc(results)[:top_k]
        relevant_doc_ids = case["relevant_doc_ids"]

        ap_scores.append(average_precision_at_k(predicted_doc_ids, relevant_doc_ids, k=top_k))
        rr_scores.append(reciprocal_rank_at_k(predicted_doc_ids, relevant_doc_ids, k=top_k))
        ndcg_scores.append(ndcg_at_k(predicted_doc_ids, relevant_doc_ids, k=top_k))

    return {
        "retriever": mode,
        f"MAP@{top_k}": float(np.mean(ap_scores)) if ap_scores else 0.0,
        f"MRR@{top_k}": float(np.mean(rr_scores)) if rr_scores else 0.0,
        f"nDCG@{top_k}": float(np.mean(ndcg_scores)) if ndcg_scores else 0.0,
        "num_queries": len(valid_eval_cases),
    }

print("\nEvaluating retrieval methods...")
retrieval_modes = ["bm25", "semantic", "hybrid_rrf", "hybrid_score"]

# Added comment: Evaluate every retrieval mode on the same set of questions for a fair comparison.
evaluation_results = [
    evaluate_retriever(mode, top_k=EVALUATION_TOP_K)
    for mode in retrieval_modes
]

evaluation_df = pd.DataFrame(evaluation_results)
display(evaluation_df)

print("\nInterpretation:")
print("- MAP@10 rewards retrieving relevant documents anywhere in the top 10, with higher reward for higher rank.")
print("- MRR@10 focuses on the rank of the first relevant document.")
print("- nDCG@10 rewards good ranking quality and is commonly used for search evaluation.")



Evaluation queries with matched relevant documents: 10

Evaluating retrieval methods...


,retriever,MAP@10,MRR@10,nDCG@10,num_queries
0,bm25,0.858333,0.858333,0.893068,10
1,semantic,1.000000,1.000000,1.000000,10
2,hybrid_rrf,0.900000,0.900000,0.926186,10
3,hybrid_score,0.950000,0.950000,0.963093,10



Interpretation:
- MAP@10 rewards retrieving relevant documents anywhere in the top 10, with higher reward for higher rank.
- MRR@10 focuses on the rank of the first relevant document.
- nDCG@10 rewards good ranking quality and is commonly used for search evaluation.


## **Milestone 3: *Compare BM25, Semantic, and Hybrid Search Outputs (Part 3)***
**Description:** This section displays sample results from the hybrid retrieval system. It runs the same sample questions using hybrid search and shows the top 10 retrieved passages. This allows the user to visually compare how hybrid retrieval improves or changes the ranking compared with BM25 alone.

In [109]:
# ============================================================
# Inspect hybrid search results
# ============================================================
# Added comment: This cell shows example outputs from hybrid retrieval.
# Added comment: It helps compare hybrid search behavior against the earlier BM25-only results.


print("\nHybrid RRF top-10 results for sample questions:")
for question in sample_questions:
    print("\n" + "=" * 90)
    print(f"Question: {question}")
    # Added comment: Use hybrid_rrf here because it is the default balanced retrieval strategy.
    hybrid_results = retrieve(question, top_k=10, mode="hybrid_rrf")
    display(results_to_dataframe(hybrid_results))



Hybrid RRF top-10 results for sample questions:

Question: What is the difference between artificial intelligence and machine learning?


,rank,method,score,source,title,tags,link,text_preview
0,1,hybrid_rrf,0.03252,Data Science StackExchange,Difference between machine learning and artifi...,"machine-learning, ai, theory",https://datascience.stackexchange.com/question...,Title: Difference between machine learning and...
1,2,hybrid_rrf,0.03202,AI StackExchange,What is the difference between artificial inte...,"machine-learning, comparison, terminology, ai-...",https://ai.stackexchange.com/questions/35/what...,Title: What is the difference between artifici...
2,3,hybrid_rrf,0.03175,AI StackExchange,What is the difference between artificial inte...,"terminology, comparison, cognitive-science",https://ai.stackexchange.com/questions/1847/wh...,Title: What is the difference between artifici...
3,4,hybrid_rrf,0.03151,AI StackExchange,What is the difference between artificial inte...,"terminology, robotics, robots, comparison",https://ai.stackexchange.com/questions/1462/wh...,Title: What is the difference between artifici...
4,5,hybrid_rrf,0.03078,AI StackExchange,What is the difference between artificial inte...,"comparison, terminology, definitions",https://ai.stackexchange.com/questions/7446/wh...,Title: What is the difference between artifici...
5,6,hybrid_rrf,0.02964,AI StackExchange,What is the difference between machine learnin...,"machine-learning, deep-learning, comparison",https://ai.stackexchange.com/questions/1742/wh...,Title: What is the difference between machine ...
6,7,hybrid_rrf,0.02859,AI StackExchange,Is artificial intelligence really just human i...,"philosophy, intelligence, artificial-creativity",https://ai.stackexchange.com/questions/16646/i...,Title: Is artificial intelligence really just ...
7,8,hybrid_rrf,0.02604,AI StackExchange,Is a genetic algorithm an example of artificia...,"philosophy, genetic-algorithms, terminology",https://ai.stackexchange.com/questions/28/is-a...,Title: Is a genetic algorithm an example of ar...
8,9,hybrid_rrf,0.02599,AI StackExchange,"What are good alternatives to the expression ""...","philosophy, terminology",https://ai.stackexchange.com/questions/3176/wh...,Title: What are good alternatives to the expre...
9,10,hybrid_rrf,0.02556,AI StackExchange,How is Bayes' Theorem used in artificial intel...,"machine-learning, applications, bayes-theorem",https://ai.stackexchange.com/questions/2738/ho...,Title: How is Bayes' Theorem used in artificia...



Question: How can neural networks handle varying input sizes?


,rank,method,score,source,title,tags,link,text_preview
0,1,hybrid_rrf,0.03279,AI StackExchange,How can neural networks deal with varying inpu...,"neural-networks, machine-learning, deep-learni...",https://ai.stackexchange.com/questions/2008/ho...,Title: How can neural networks deal with varyi...
1,2,hybrid_rrf,0.03058,Data Science StackExchange,How to prepare the varied size input in CNN pr...,"machine-learning, neural-network, deep-learnin...",https://datascience.stackexchange.com/question...,Title: How to prepare the varied size input in...
2,3,hybrid_rrf,0.03033,AI StackExchange,How to handle rectangular images in convolutio...,"neural-networks, convolutional-neural-networks...",https://ai.stackexchange.com/questions/8323/ho...,Title: How to handle rectangular images in con...
3,4,hybrid_rrf,0.03012,AI StackExchange,Why do we need floats for using neural networks?,"neural-networks, machine-learning",https://ai.stackexchange.com/questions/7247/wh...,Title: Why do we need floats for using neural ...
4,5,hybrid_rrf,0.02872,AI StackExchange,Are there neural networks with very few nodes ...,neural-networks,https://ai.stackexchange.com/questions/10944/a...,Title: Are there neural networks with very few...
5,6,hybrid_rrf,0.02712,Data Science StackExchange,How to feed LSTM with different input array si...,"keras, lstm",https://datascience.stackexchange.com/question...,Title: How to feed LSTM with different input a...
6,7,hybrid_rrf,0.02642,AI StackExchange,Can a single neural network handle recognizing...,"neural-networks, image-recognition",https://ai.stackexchange.com/questions/13/can-...,"need for rapid, real-time processing, the maxi..."
7,8,hybrid_rrf,0.02639,AI StackExchange,How can Transformers handle arbitrary length i...,"natural-language-processing, recurrent-neural-...",https://ai.stackexchange.com/questions/22957/h...,Title: How can Transformers handle arbitrary l...
8,9,hybrid_rrf,0.02439,AI StackExchange,How could we build a neural network that is in...,"neural-networks, machine-learning, reference-r...",https://ai.stackexchange.com/questions/4655/ho...,Title: How could we build a neural network tha...
9,10,hybrid_rrf,0.02427,AI StackExchange,Which functions can't neural networks learn ef...,"neural-networks, machine-learning, reference-r...",https://ai.stackexchange.com/questions/5539/wh...,Title: Which functions can't neural networks l...



Question: What is the difference between fit and fit_transform in scikit-learn?


,rank,method,score,source,title,tags,link,text_preview
0,1,hybrid_rrf,0.03279,Data Science StackExchange,What's the difference between fit and fit_tran...,"python, scikit-learn",https://datascience.stackexchange.com/question...,Title: What's the difference between fit and f...
1,2,hybrid_rrf,0.03226,Data Science StackExchange,What is the difference between fit() and fit_g...,"deep-learning, keras",https://datascience.stackexchange.com/question...,Title: What is the difference between fit() an...
2,3,hybrid_rrf,0.02828,Data Science StackExchange,Difference between OrdinalEncoder and LabelEnc...,"machine-learning, python, scikit-learn, prepro...",https://datascience.stackexchange.com/question...,Title: Difference between OrdinalEncoder and L...
3,4,hybrid_rrf,0.02786,Data Science StackExchange,What is the difference between CountVectorizer...,"machine-learning, nlp, scikit-learn",https://datascience.stackexchange.com/question...,Title: What is the difference between CountVec...
4,5,hybrid_rrf,0.02783,Data Science StackExchange,How can I fit categorical data types for rando...,"python, scikit-learn, data-mining, random-forest",https://datascience.stackexchange.com/question...,Title: How can I fit categorical data types fo...
5,6,hybrid_rrf,0.02698,Data Science StackExchange,Data normalization before or after train-test ...,normalization,https://datascience.stackexchange.com/question...,Title: Data normalization before or after trai...
6,7,hybrid_rrf,0.02671,Data Science StackExchange,Can you explain the difference between SVC and...,"svm, scikit-learn",https://datascience.stackexchange.com/question...,Title: Can you explain the difference between ...
7,8,hybrid_rrf,0.02663,Data Science StackExchange,When to use One Hot Encoding vs LabelEncoder v...,"scikit-learn, categorical-data, feature-engine...",https://datascience.stackexchange.com/question...,Title: When to use One Hot Encoding vs LabelEn...
8,9,hybrid_rrf,0.02642,Data Science StackExchange,What is the difference between cross_validate ...,"machine-learning, scikit-learn",https://datascience.stackexchange.com/question...,Title: What is the difference between cross_va...
9,10,hybrid_rrf,0.02590,Data Science StackExchange,What's the difference between Sklearn F1 score...,"classification, scikit-learn, metric",https://datascience.stackexchange.com/question...,Title: What's the difference between Sklearn F...


## **Milestone 4: *RAG Prompt and Citation Formatting (Part 1)***
**Description:** This section prepares retrieved passages for answer generation. It labels each retrieved source as `[S1]`, `[S2]`, `[S3]`, and so on, fetches additional StackExchange answer evidence when available, and builds a structured RAG context. It also formats retrieved sources into a table so that the generated answer can be checked against its citations.

In [110]:
# ============================================================
# RAG Prompt and Citation Formatting
# ============================================================
# Added comment: This cell prepares retrieved sources for RAG by adding citation labels and answer evidence.
# Added comment: It also fetches StackExchange answer text when available to improve generation quality.

#
# This section prepares retrieved sources for answer generation.
#
# Important:
# The original JSON corpus mostly contains StackExchange question posts.
# To generate better factual answers, this section can enrich the retrieved
# question sources by fetching their StackExchange answers using the public
# StackExchange API.
#
# The final answer will then be based on retrieved sources and their answer text,
# not only on question titles.

import requests
import time

# Added comment: Cache API results so repeated questions do not keep requesting the same answers.
STACKEXCHANGE_ANSWER_CACHE = {}

def clean_stackexchange_html(raw_html):
    """
    Convert StackExchange HTML answer text into clean plain text.
    """
    if raw_html is None:
        return ""

    soup = BeautifulSoup(raw_html, "html.parser")

    for code_tag in soup.find_all(["code", "pre"]):
        code_text = code_tag.get_text(" ", strip=True)
        code_tag.replace_with(f" CODE_SNIPPET {code_text} ")

    text = soup.get_text(" ", strip=True)
    text = html.unescape(text)
    text = re.sub(r"\s+", " ", text).strip()

    return text


def get_stackexchange_site_from_link(link):
    """
    Identify the correct StackExchange API site name from the source link.
    """
    link = str(link).lower()

    if "ai.stackexchange.com" in link:
        return "ai"

    if "datascience.stackexchange.com" in link:
        return "datascience"

    return None


def get_question_id_from_link(link):
    """
    Extract the StackExchange question ID from a question link.
    """
    match = re.search(r"/questions/(\d+)", str(link))

    if match:
        return match.group(1)

    return None


def fetch_stackexchange_answers(link, max_answers=2):
    """
    Fetch accepted/top answers for a StackExchange question.

    This improves answer generation because the original JSON corpus mainly
    contains question posts, while the public API can provide answer bodies.

    If the API request fails, the function safely returns an empty list.
    """
    # Added comment: The API requires the site name and question ID extracted from the source URL.
    site = get_stackexchange_site_from_link(link)
    question_id = get_question_id_from_link(link)

    if site is None or question_id is None:
        return []

    cache_key = f"{site}_{question_id}"

    # Added comment: Return cached answers instantly if this question was already fetched.
    if cache_key in STACKEXCHANGE_ANSWER_CACHE:
        return STACKEXCHANGE_ANSWER_CACHE[cache_key]

    api_url = f"https://api.stackexchange.com/2.3/questions/{question_id}/answers"

    params = {
        "site": site,
        "order": "desc",
        "sort": "votes",
        "filter": "withbody",
        "pagesize": max_answers,
    }

    try:
        response = requests.get(api_url, params=params, timeout=15)

        if response.status_code != 200:
            STACKEXCHANGE_ANSWER_CACHE[cache_key] = []
            return []

        data = response.json()
        items = data.get("items", [])

        # Added comment: Store cleaned answer bodies with score metadata before ranking them.
        answers = []

        for item in items:
            answer_text = clean_stackexchange_html(item.get("body", ""))

            if not answer_text:
                continue

            answers.append({
                "answer_id": item.get("answer_id"),
                "is_accepted": item.get("is_accepted", False),
                "score": item.get("score", 0),
                "answer_text": answer_text,
            })

        # Prefer accepted answers first, then high-score answers.
        # Added comment: Accepted and high-scoring answers are preferred because they are more likely to be useful.
        answers = sorted(
            answers,
            key=lambda x: (x["is_accepted"], x["score"]),
            reverse=True,
        )

        STACKEXCHANGE_ANSWER_CACHE[cache_key] = answers[:max_answers]

        # Small delay to be polite to the public API.
        time.sleep(0.1)

        return STACKEXCHANGE_ANSWER_CACHE[cache_key]

    except Exception:
        STACKEXCHANGE_ANSWER_CACHE[cache_key] = []
        return []


def enrich_sources_with_answers(used_sources, max_answer_chars=1800):
    """
    Add answer text to each retrieved source when available.

    Each source keeps:
    - title
    - retrieved question passage
    - fetched answer text, if available
    """
    enriched_sources = []

    for source in used_sources:
        enriched = dict(source)

        answers = fetch_stackexchange_answers(source["link"], max_answers=2)

        answer_parts = []

        for answer in answers:
            answer_text = answer["answer_text"]

            if len(answer_text) > max_answer_chars:
                answer_text = answer_text[:max_answer_chars] + "..."

            accepted_label = "Accepted answer" if answer["is_accepted"] else "High-scoring answer"

            answer_parts.append(
                f"{accepted_label}, score {answer['score']}: {answer_text}"
            )

        enriched["answer_text"] = "\n".join(answer_parts).strip()

        enriched_sources.append(enriched)

    return enriched_sources


def build_rag_context(results, max_chars=MAX_CONTEXT_CHARS):
    """
    Build the cited RAG context.

    Each source is labeled as [S1], [S2], [S3], etc.
    The context includes fetched answer text when available.
    """
    used_sources = []

    for source_number, result in enumerate(results, start=1):
        used_source = dict(result)
        used_source["source_label"] = f"S{source_number}"
        used_sources.append(used_source)

    # Added comment: Enrich the retrieved sources before constructing the final RAG context.
    enriched_sources = enrich_sources_with_answers(used_sources)

    context_parts = []
    current_chars = 0

    for source in enriched_sources:
        label = source["source_label"]

        retrieved_passage = re.sub(r"\s+", " ", source["text"]).strip()

        # Added comment: Prefer answer evidence; fall back to retrieved question text if no answer is available.
        if source.get("answer_text"):
            evidence_text = source["answer_text"]
            evidence_type = "StackExchange answer evidence"
        else:
            evidence_text = retrieved_passage
            evidence_type = "Retrieved question passage only"

        source_block = (
            f"[{label}]\n"
            f"Title: {source['title']}\n"
            f"Source: {source['source']}\n"
            f"Tags: {source['tags']}\n"
            f"URL: {source['link']}\n"
            f"Evidence type: {evidence_type}\n"
            f"Evidence: {evidence_text}\n"
        )

        if current_chars + len(source_block) > max_chars:
            break

        context_parts.append(source_block)
        current_chars += len(source_block)

    return "\n\n".join(context_parts), enriched_sources


def build_rag_prompt(question, context):
    """
    Build the RAG prompt used by OpenAI if an API key is available.
    """
    # Added comment: The prompt explicitly tells the generator how to cite and avoid unsupported claims.
    prompt = f"""
You are a technical question-answering assistant.

Answer the user's question clearly and factually using the retrieved sources.

Rules:
- Write a direct answer in 4 to 7 sentences.
- Use citations like [S1], [S2], and [S3].
- Only cite sources that support the answer.
- Do not output only citations.
- Do not copy source text blindly.
- If the retrieved evidence is limited, say so briefly.
- End with a line called "Sources used:".

User question:
{question}

Retrieved sources:
{context}

Final answer:
""".strip()

    return prompt


def sources_to_dataframe(used_sources):
    """
    Convert retrieved/enriched sources to a table for Gradio.
    """
    # Added comment: The source table lets the user audit exactly where the answer came from.
    rows = []

    for source in used_sources:
        answer_preview = source.get("answer_text", "")

        if answer_preview:
            preview = answer_preview[:260]
        else:
            preview = source["text"][:260]

        if len(preview) >= 260:
            preview += "..."

        rows.append({
            "citation": f"[{source['source_label']}]",
            "rank": source["rank"],
            "score": round(float(source["score"]), 5),
            "source": source["source"],
            "title": source["title"],
            "tags": source["tags"],
            "link": source["link"],
            "preview": preview,
        })

    return pd.DataFrame(rows)


## **Milestone 4: *RAG Answer Generation (Part 2)***
**Description:** This section generates the final answer using the retrieved and cited sources. If an OpenAI API key is available, the notebook can use OpenAI for fluent RAG generation. If no API key is available, the notebook uses a local rule-based and evidence-based answer generator that creates clean cited answers from the retrieved sources without producing messy copied fragments.

In [111]:
# ============================================================
# Milestone 4: RAG Answer Generation
# ============================================================
# Added comment: This cell generates the final cited answer shown to the user.
# Added comment: It uses OpenAI when available, otherwise it uses the local Qwen instruction model.
# Added comment: The cleaning and validation functions prevent raw copied fragments from appearing in the final answer.

#
# This section generates the final cited answer.
#
# Important fix:
# FLAN-T5 was producing weak, short, and incoherent answers.
# This version uses a stronger local instruction model:
#
#     Qwen/Qwen2.5-3B-Instruct
#
# This model is much better for generating clean explanatory answers.
#
# Final answer format:
# - Clear generated paragraph
# - Citations such as [S1], [S2], [S3]
# - Final "Sources used:" line

from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import re
import os

# Added comment: Qwen is used as the local instruction model for cleaner generation than earlier FLAN tests.
LOCAL_GENERATION_MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

local_generation_tokenizer = None
local_generation_model = None


def openai_key_available():
    """
    Check whether an OpenAI API key is available.
    """
    return bool(os.environ.get("OPENAI_API_KEY", "").strip())


def generate_with_openai(prompt, temperature=0.1):
    """
    Generate a cited answer using OpenAI if an API key is available.
    """
    from openai import OpenAI

    client = OpenAI()

    response = client.chat.completions.create(
        model=OPENAI_MODEL,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a careful technical RAG assistant. "
                    "Answer clearly, factually, and cite retrieved sources."
                ),
            },
            {
                "role": "user",
                "content": prompt,
            },
        ],
        temperature=temperature,
    )

    return response.choices[0].message.content.strip()


def load_local_generation_model():
    """
    Load the local Qwen instruction model once.

    This model should work well on A100/T4 GPU.
    First load may take a few minutes.
    """
    global local_generation_tokenizer, local_generation_model

    if local_generation_tokenizer is not None and local_generation_model is not None:
        return local_generation_tokenizer, local_generation_model

    print(f"Loading local generation model: {LOCAL_GENERATION_MODEL_NAME}")
    print("This may take a few minutes the first time.")

    local_generation_tokenizer = AutoTokenizer.from_pretrained(
        LOCAL_GENERATION_MODEL_NAME,
        trust_remote_code=True
    )

    # Added comment: bfloat16 reduces GPU memory use on supported hardware while keeping generation quality.
    dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32

    local_generation_model = AutoModelForCausalLM.from_pretrained(
        LOCAL_GENERATION_MODEL_NAME,
        torch_dtype=dtype,
        device_map="auto",
        trust_remote_code=True
    )

    local_generation_model.eval()

    device_name = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Local generation model loaded on: {device_name}")

    return local_generation_tokenizer, local_generation_model


def clean_source_text_for_generation(text):
    """
    Clean retrieved/fetched source text before giving it to the generator.
    """
    text = str(text)

    text = re.sub(
        r"(Accepted answer|High-scoring answer),?\s*score\s*\d+\s*:\s*",
        "",
        text,
        flags=re.IGNORECASE,
    )

    text = re.sub(r"https?://\S+", "", text)
    text = re.sub(r"www\.\S+", "", text)

    text = text.replace("CODE_SNIPPET", "code snippet")

    text = re.sub(r"\s+", " ", text).strip()

    return text


def build_generation_context(enriched_sources, max_chars=6500):
    """
    Build a clean context for the local instruction model.

    Each source keeps its citation label.
    """
    context_blocks = []
    current_chars = 0

    for source in enriched_sources:
        label = source["source_label"]

        # Added comment: Use StackExchange answer evidence first because it is more explanatory than question text.
        evidence = source.get("answer_text", "")

        if not evidence:
            evidence = source.get("text", "")

        evidence = clean_source_text_for_generation(evidence)

        # Keep each source compact so the model does not get overloaded.
        evidence = evidence[:1400]

        # Added comment: Keep the citation label next to each evidence block so the model can cite it.
        block = (
            f"[{label}]\n"
            f"Title: {source['title']}\n"
            f"Evidence: {evidence}\n"
        )

        if current_chars + len(block) > max_chars:
            break

        context_blocks.append(block)
        current_chars += len(block)

    return "\n".join(context_blocks)


def build_local_generation_prompt(question, enriched_sources):
    """
    Build a strict prompt for the local instruction model.

    The prompt forces the model to:
    - answer clearly,
    - avoid copying raw fragments,
    - cite sources,
    - avoid unsupported claims.
    """
    context = build_generation_context(
        enriched_sources=enriched_sources,
        max_chars=6500,
    )

    prompt = f"""
You are a technical question-answering assistant.

Answer the user's question using ONLY the cited sources below.

User question:
{question}

Cited sources:
{context}

Instructions:
- Write a clear, factual answer in 4 to 6 sentences.
- Do not copy raw source fragments.
- Do not mention answer scores, accepted answers, or source metadata.
- Use citations like [S1], [S2], and [S3] inside the answer.
- Only cite sources that support the sentence.
- Do not invent information that is not supported by the sources.
- End with this exact format:
Sources used: [S1], [S2]

Final answer:
""".strip()

    return prompt


def generate_with_local_model(question, enriched_sources):
    """
    Generate a cited answer using Qwen2.5-3B-Instruct.
    """
    tokenizer, model = load_local_generation_model()

    prompt = build_local_generation_prompt(
        question=question,
        enriched_sources=enriched_sources,
    )

    # Added comment: Chat-template messages make the instruction model behave like a helpful assistant.
    messages = [
        {
            "role": "system",
            "content": (
                "You are a careful RAG assistant. "
                "You answer clearly and cite only the provided sources."
            ),
        },
        {
            "role": "user",
            "content": prompt,
        },
    ]

    try:
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
    except Exception:
        text = prompt

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=4096,
    ).to(model.device)

    with torch.no_grad():
        # Added comment: Deterministic decoding is used so demo answers stay stable between runs.
        output_ids = model.generate(
            **inputs,
            max_new_tokens=320,
            do_sample=False,
            temperature=0.1,
            top_p=0.9,
            repetition_penalty=1.08,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated_ids = output_ids[0][inputs["input_ids"].shape[-1]:]

    answer = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True,
    )

    return answer.strip()


def clean_final_answer(answer):
    """
    Clean final answer before displaying it.
    """
    answer = str(answer).strip()

    # Remove accidental source metadata.
    answer = re.sub(
        r"(Accepted answer|High-scoring answer),?\s*score\s*\d+\s*:\s*",
        "",
        answer,
        flags=re.IGNORECASE,
    )

    # Remove raw URLs.
    answer = re.sub(r"https?://\S+", "", answer)
    answer = re.sub(r"www\.\S+", "", answer)

    # Remove accidental prompt leakage.
    unwanted_prefixes = [
        "Final answer:",
        "Answer:",
        "Assistant:",
    ]

    for prefix in unwanted_prefixes:
        if answer.lower().startswith(prefix.lower()):
            answer = answer[len(prefix):].strip()

    answer = re.sub(r"\s+", " ", answer).strip()

    # Put Sources used on a separate line.
    answer = re.sub(r"\s*(Sources used:)", r"\n\n\1", answer)

    return answer


def is_bad_generated_answer(answer):
    """
    Detect invalid or messy generated answers.
    """
    if answer is None:
        return True

    answer = str(answer).strip()

    if len(answer) < 120:
        return True

    answer_lower = answer.lower()

    bad_phrases = [
        "accepted answer, score",
        "high-scoring answer, score",
        "see pretty nice blog",
        "youtube video",
        "taking excerpts from",
        "click here",
        "i cannot answer",
        "i don't have enough information",
    ]

    for phrase in bad_phrases:
        if phrase in answer_lower:
            return True

    # Must include at least one source citation.
    if not re.search(r"\[S\d+\]", answer):
        return True

    # Detect citation-only answer.
    citation_removed = re.sub(r"\[S\d+\]", "", answer)
    citation_removed = re.sub(r"[\s,.;:()\[\]-]", "", citation_removed)

    if citation_removed == "":
        return True

    return False


def build_clean_fallback_answer(question, enriched_sources):
    """
    Clean fallback only if generation fails completely.
    """
    labels = []

    for source in enriched_sources[:3]:
        label = f"[{source['source_label']}]"

        if label not in labels:
            labels.append(label)

    return (
        "The system retrieved relevant cited sources, but the local generator could not produce "
        "a reliable fluent answer. Please review the retrieved source table for the supporting evidence.\n\n"
        f"Sources used: {', '.join(labels)}"
    )


def ensure_sources_used_line(answer, enriched_sources):
    """
    Ensure the answer ends with a Sources used line.
    """
    if "Sources used:" in answer:
        return answer

    citations = re.findall(r"\[S\d+\]", answer)

    if not citations:
        citations = [
            f"[{source['source_label']}]"
            for source in enriched_sources[:3]
        ]

    unique_citations = []

    for citation in citations:
        if citation not in unique_citations:
            unique_citations.append(citation)

    return answer.strip() + "\n\nSources used: " + ", ".join(unique_citations)


def generate_answer(
    question,
    retrieval_mode="hybrid_rrf",
    top_k=DEFAULT_TOP_K,
    generator_choice="auto",
    temperature=0.1,
):
    """
    Full RAG answer-generation pipeline.

    Steps:
    1. Retrieve relevant passages.
    2. Enrich retrieved passages with StackExchange answer evidence.
    3. Generate a cited answer using OpenAI if available.
    4. Otherwise generate a cited answer using Qwen2.5-3B-Instruct.
    5. Clean and validate the answer.
    6. Return both the answer and retrieved sources.
    """
    if not question or not question.strip():
        return "Please enter a question.", []

    results = retrieve(
        query=question,
        top_k=top_k,
        mode=retrieval_mode,
    )

    context, enriched_sources = build_rag_context(
        results=results,
        max_chars=MAX_CONTEXT_CHARS,
    )

    if not enriched_sources:
        return "I could not retrieve any relevant sources for this question.", []

    openai_prompt = build_rag_prompt(
        question=question,
        context=context,
    )

    generator_choice = generator_choice.lower().strip()

    try:
        if generator_choice in ["openai", "auto"] and openai_key_available():
            answer = generate_with_openai(
                prompt=openai_prompt,
                temperature=temperature,
            )
        else:
            answer = generate_with_local_model(
                question=question,
                enriched_sources=enriched_sources,
            )

        # Added comment: Clean accidental URLs/metadata before validating and displaying the answer.
        answer = clean_final_answer(answer)
        answer = ensure_sources_used_line(answer, enriched_sources)

        # Added comment: If the model output is unusable, return a clean fallback instead of messy text.
        if is_bad_generated_answer(answer):
            answer = build_clean_fallback_answer(
                question=question,
                enriched_sources=enriched_sources,
            )

        return answer, enriched_sources

    except Exception as error:
        print(f"Generation warning: {error}")

        answer = build_clean_fallback_answer(
            question=question,
            enriched_sources=enriched_sources,
        )

        return answer, enriched_sources


## **Milestone 5: *Citation Quality Evaluation***
**Description:** This section checks whether the generated answer uses citations correctly. It detects citation labels such as `[S1]`, verifies that they match retrieved sources, and reports whether any invalid citations were used. This helps confirm that the answer is grounded in the retrieved evidence rather than unsupported text.

In [112]:
# ============================================================
# Citation quality checking
# ============================================================
# Added comment: This cell checks whether generated answers include valid citations from the retrieved sources.
# Added comment: It is useful for proving that the answer is grounded in the displayed source table.


def check_citation_quality(answer: str, used_sources: List[Dict[str, Any]]) -> Dict[str, Any]:
    """
    Basic automatic citation quality check.

    This checks:
    - Did the answer include citations?
    - Are the cited source IDs valid?
    - Did it cite at least one retrieved source?
    """
    # Added comment: Extract every citation label that appears in the generated answer.
    cited_labels = re.findall(r"\[S\d+\]", answer)
    # Added comment: Valid citations are only the labels assigned to the retrieved sources.
    valid_labels = {f"[{s['source_label']}]" for s in used_sources}

    invalid_labels = sorted(set(cited_labels) - valid_labels)
    valid_cited_labels = sorted(set(cited_labels) & valid_labels)

    return {
        "has_any_citation": len(cited_labels) > 0,
        "valid_citations_used": ", ".join(valid_cited_labels) if valid_cited_labels else "None",
        "invalid_citations": ", ".join(invalid_labels) if invalid_labels else "None",
        "num_sources_retrieved": len(used_sources),
    }

# Added comment: Optional demo run shows one generated answer and validates its citations.
if RUN_DEMO_RAG_ANSWER:
    demo_question = "What is the difference between artificial intelligence and machine learning?"

    print("\n" + "=" * 90)
    print("Demo RAG answer")
    print("=" * 90)
    print(f"Question: {demo_question}")

    demo_answer, demo_sources = generate_answer(
        question=demo_question,
        retrieval_mode="hybrid_rrf",
        top_k=5,
        generator_choice="auto",
        temperature=0.1,
    )

    display(Markdown(demo_answer))

    print("\nRetrieved sources:")
    display(sources_to_dataframe(demo_sources))

    print("\nCitation quality check:")
    display(pd.DataFrame([check_citation_quality(demo_answer, demo_sources)]))



Demo RAG answer
Question: What is the difference between artificial intelligence and machine learning?
Loading local generation model: Qwen/Qwen2.5-3B-Instruct
This may take a few minutes the first time.


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Local generation model loaded on: cuda


Artificial Intelligence (AI) encompasses a broader range of technologies and methodologies aimed at creating intelligent machines that can perform tasks requiring human-like cognition, including reasoning, problem-solving, perception, and learning. Machine Learning (ML), on the other hand, is a specific subset within AI focused on enabling computers to automatically improve their performance on a specific task through exposure to data, without explicit programming. While ML involves techniques like supervised, unsupervised, and reinforcement learning, these methods are part of a larger AI framework that includes other approaches such as symbolic AI, neural networks, and evolutionary algorithms. Therefore, AI provides the overarching concept and potential applications, whereas ML represents a particular approach within AI dedicated to achieving intelligent behavior through data-driven improvements.

Sources used: [S1], [S2]


Retrieved sources:


,citation,rank,score,source,title,tags,link,preview
0,[S1],1,0.03252,Data Science StackExchange,Difference between machine learning and artifi...,"machine-learning, ai, theory",https://datascience.stackexchange.com/question...,"High-scoring answer, score 10: The subject are..."
1,[S2],2,0.03202,AI StackExchange,What is the difference between artificial inte...,"machine-learning, comparison, terminology, ai-...",https://ai.stackexchange.com/questions/35/what...,"High-scoring answer, score 65: Machine learnin..."
2,[S3],3,0.03175,AI StackExchange,What is the difference between artificial inte...,"terminology, comparison, cognitive-science",https://ai.stackexchange.com/questions/1847/wh...,"Accepted answer, score 3: Another impression t..."
3,[S4],4,0.03151,AI StackExchange,What is the difference between artificial inte...,"terminology, robotics, robots, comparison",https://ai.stackexchange.com/questions/1462/wh...,"Accepted answer, score 9: Although there are s..."
4,[S5],5,0.03078,AI StackExchange,What is the difference between artificial inte...,"comparison, terminology, definitions",https://ai.stackexchange.com/questions/7446/wh...,"High-scoring answer, score 6: What is the diff..."



Citation quality check:


,has_any_citation,valid_citations_used,invalid_citations,num_sources_retrieved
0,True,"[S1], [S2]",None,5


#
---

# **UI Integration: *Gradio App***
**Description:** This section builds the interactive Gradio interface for live testing and demonstration. The user can type a question, choose the retrieval mode, select the number of retrieved sources, choose the generator mode, and generate an answer. The interface displays both the cited answer and a table of retrieved sources, making the system easy to test during the project demo.

In [113]:
# ============================================================
# UI Integration: Gradio App
# ============================================================
# Added comment: This cell creates the Gradio web interface used for the final live demo.
# Added comment: The UI connects user input to retrieval, generation, citation display, and the source table.

#
# This section builds the interactive Gradio interface.
#
# Design:
# - Centered title card.
# - Intro/features card.
# - Inputs placed on the left.
# - Generated answer placed on the right.
# - Retrieved sources displayed underneath.
# - Examples placed inside a clean section.
# - 10 example questions included for testing.

from IPython.display import display, HTML

def gradio_answer(question, retrieval_mode, top_k, generator_choice, temperature):
    """
    Run the complete Q&A pipeline from the Gradio interface.

    Inputs:
    - question: The user's technical question.
    - retrieval_mode: BM25, semantic, hybrid_rrf, or hybrid_score.
    - top_k: Number of retrieved sources.
    - generator_choice: auto, local, or openai.
    - temperature: OpenAI generation temperature.

    Outputs:
    - answer: Generated answer with citations.
    - source_df: Retrieved source passages in table format.
    """

    # Added comment: This single call runs retrieval, source enrichment, generation, and citation handling.
    answer, used_sources = generate_answer(
        question=question,
        retrieval_mode=retrieval_mode,
        top_k=int(top_k),
        generator_choice=generator_choice,
        temperature=float(temperature),
    )

    # Added comment: Convert source dictionaries into a table for display under the answer.
    source_df = sources_to_dataframe(used_sources)

    return answer, source_df


# ------------------------------------------------------------
# Close any previously running Gradio app
# ------------------------------------------------------------
# This prevents duplicate Gradio apps from appearing when rerunning the cell.

try:
    demo.close()
except Exception:
    pass


# ------------------------------------------------------------
# Custom CSS for cleaner presentation-style layout
# ------------------------------------------------------------

# Added comment: CSS controls the visual layout without changing the Q&A pipeline logic.
CUSTOM_CSS = """
/* ============================================================
   Main page styling
   ============================================================ */

/* Use full browser width while controlling internal layout width */
.gradio-container {
    max-width: 100% !important;
    width: 100% !important;
    margin: 0 auto !important;
}

/* Overall dark background */
body,
.gradio-container {
    background: #101114 !important;
}

/* Uniform layout width for all sections */
.clean-section {
    max-width: 1400px;
    width: 92%;
    margin: 0 auto 16px auto;
}

/* Header card */
#header-card {
    background: #17181c;
    border: 1px solid #33363d;
    border-radius: 12px;
    padding: 26px 24px;
    text-align: center;
    margin-top: 20px;
}

/* Main title */
#header-card h1 {
    font-weight: 900 !important;
    font-size: 40px !important;
    text-align: center !important;
    margin: 0 0 10px 0 !important;
}

/* Subtitle */
#header-card h2 {
    font-weight: 700 !important;
    font-size: 22px !important;
    text-align: center !important;
    margin: 0 !important;
}

/* Intro card */
#intro-card {
    background: #17181c;
    border: 1px solid #33363d;
    border-radius: 12px;
    padding: 22px 24px;
    margin-bottom: 16px;
}

/* Intro text readability */
#intro-card p,
#intro-card li {
    font-size: 15px !important;
    line-height: 1.55 !important;
}

/* Remove large grey panel feeling */
.clean-section,
.main-output-row {
    background: transparent !important;
}

/* Make individual Gradio components cleaner */
.block,
.gr-box,
.form {
    background: #17181c !important;
    border-color: #33363d !important;
}

/* Question box readability */
#question-box textarea {
    font-size: 15px !important;
    line-height: 1.5 !important;
}

/* Answer box readability */
#answer-box {
    font-size: 15px !important;
    line-height: 1.55 !important;
}

/* Left and right layout sizing */
#left-control-column {
    min-width: 320px;
}

#right-answer-column {
    min-width: 600px;
}

/* Orange primary button */
button.primary {
    background: #f97316 !important;
    border-color: #f97316 !important;
}

/* Improve spacing */
.contain {
    gap: 14px !important;
}

/* Dataframe section */
#source-table {
    max-height: 500px;
}
"""


# ------------------------------------------------------------
# Build Gradio interface
# ------------------------------------------------------------

# Added comment: Gradio Blocks lets us arrange the interface into cards, rows, columns, and tables.
with gr.Blocks(
    title=PROJECT_NAME,
    fill_width=True,
    css=CUSTOM_CSS
) as demo:

    # ------------------------------------------------------------
    # Header section
    # ------------------------------------------------------------

    with gr.Column(elem_classes="clean-section"):

        gr.Markdown(
            """
            <div id="header-card">

            # Smart Q&A Assistant
            ## Hybrid Search with Cited Technical Answers

            </div>
            """
        )

    # ------------------------------------------------------------
    # Intro section
    # ------------------------------------------------------------

    with gr.Column(elem_classes="clean-section"):

        gr.Markdown(
            """
            <div id="intro-card">

            This interface allows the user to ask a technical AI or Data Science question.

            The system retrieves relevant StackExchange-style passages using one of several retrieval methods,
            then generates an answer with citations.

            **Main features**
            - BM25 keyword retrieval
            - Semantic retrieval
            - Hybrid retrieval using reciprocal rank fusion or score fusion
            - Local or OpenAI-based generation
            - Answer generation with cited retrieved sources
            - Source table for transparency and explainability

            </div>
            """
        )

    # ------------------------------------------------------------
    # Main interaction layout
    # ------------------------------------------------------------
    # Left side:
    # - Question input
    # - Retrieval mode
    # - Top-k sources
    # - Generator choice
    # - Temperature
    # - Generate button
    #
    # Right side:
    # - Generated answer

    with gr.Row(elem_classes=["clean-section", "main-output-row"]):

        with gr.Column(scale=1, elem_id="left-control-column"):

            # Added comment: This is the main user input used as the query for retrieval.
            question_box = gr.Textbox(
                label="Your Question",
                placeholder="Example: What is the difference between artificial intelligence and machine learning?",
                lines=6,
                elem_id="question-box"
            )

            # Added comment: Users can compare lexical, semantic, and hybrid retrieval modes.
            retrieval_dropdown = gr.Dropdown(
                choices=[
                    "bm25",
                    "semantic",
                    "hybrid_rrf",
                    "hybrid_score",
                ],
                value="hybrid_rrf",
                label="Retrieval Mode",
            )

            top_k_slider = gr.Slider(
                minimum=1,
                maximum=10,
                value=5,
                step=1,
                label="Number of Retrieved Sources",
            )

            generator_dropdown = gr.Dropdown(
                choices=[
                    "auto",
                    "local",
                    "openai",
                ],
                value="auto",
                label="Generator",
            )

            temperature_slider = gr.Slider(
                minimum=0.0,
                maximum=1.0,
                value=0.1,
                step=0.1,
                label="OpenAI Temperature",
            )

            answer_button = gr.Button(
                "Generate Answer",
                variant="primary"
            )

        with gr.Column(scale=2, elem_id="right-answer-column"):

            answer_markdown = gr.Markdown(
                label="Generated Answer",
                elem_id="answer-box"
            )

    # ------------------------------------------------------------
    # Retrieved sources table
    # ------------------------------------------------------------

    with gr.Column(elem_classes="clean-section"):

        source_table = gr.Dataframe(
            label="Retrieved Sources",
            wrap=True,
            interactive=False,
            elem_id="source-table"
        )

    # ------------------------------------------------------------
    # Example questions
    # ------------------------------------------------------------

    with gr.Column(elem_classes="clean-section"):

        gr.Markdown(
            """
            ## Example Questions

            Click any example below to automatically fill the interface inputs.
            """
        )

        # Added comment: Examples make the demo easier because users can test the system with one click.
        gr.Examples(
            examples=[
                [
                    "What is the difference between artificial intelligence and machine learning?",
                    "hybrid_rrf",
                    5,
                    "auto",
                    0.1,
                ],
                [
                    "What is machine learning?",
                    "hybrid_rrf",
                    5,
                    "auto",
                    0.1,
                ],
                [
                    "How can neural networks handle varying input sizes?",
                    "hybrid_rrf",
                    5,
                    "auto",
                    0.1,
                ],
                [
                    "What is the difference between fit and fit_transform in scikit-learn?",
                    "hybrid_rrf",
                    5,
                    "auto",
                    0.1,
                ],
                [
                    "What is positional encoding in transformers?",
                    "hybrid_rrf",
                    5,
                    "auto",
                    0.1,
                ],
                [
                    "How do I handle imbalanced classes in Keras?",
                    "hybrid_rrf",
                    5,
                    "auto",
                    0.1,
                ],
                [
                    "What is the difference between gradient descent and stochastic gradient descent?",
                    "hybrid_rrf",
                    5,
                    "auto",
                    0.1,
                ],
                [
                    "What is overfitting in machine learning?",
                    "hybrid_rrf",
                    5,
                    "auto",
                    0.1,
                ],
                [
                    "What is the difference between supervised and unsupervised learning?",
                    "hybrid_rrf",
                    5,
                    "auto",
                    0.1,
                ],
                [
                    "What is cross entropy loss?",
                    "hybrid_rrf",
                    5,
                    "auto",
                    0.1,
                ],
            ],
            inputs=[
                question_box,
                retrieval_dropdown,
                top_k_slider,
                generator_dropdown,
                temperature_slider,
            ],
        )

    # ------------------------------------------------------------
    # Button action
    # ------------------------------------------------------------

    # Added comment: Connect the button to the gradio_answer function and define which outputs update.
    answer_button.click(
        fn=gradio_answer,
        inputs=[
            question_box,
            retrieval_dropdown,
            top_k_slider,
            generator_dropdown,
            temperature_slider,
        ],
        outputs=[
            answer_markdown,
            source_table,
        ],
    )


# ------------------------------------------------------------
# Launch Gradio app
# ------------------------------------------------------------

print("Launching Gradio app...")
print("In Colab, open the public gradio.live link in a new browser tab for the best full-screen view.")

demo.queue()

launch_result = demo.launch(
    share=True,
    inline=True,
    inbrowser=False,
    show_error=True,
    debug=False,
    prevent_thread_lock=True
)


# ------------------------------------------------------------
# Full-screen instruction box
# ------------------------------------------------------------

display(HTML("""
<div style="
    width: 92%;
    max-width: 1400px;
    margin: 16px auto;
    padding: 14px;
    border: 1px solid #444;
    border-radius: 10px;
    background: #111;
    color: white;
    font-family: Arial;
">
    <h3 style="margin-top: 0;">Full-Screen View</h3>
    <p>
        If the interface looks squeezed inside Colab, open the public
        <b>gradio.live</b> link printed above in a new browser tab.
    </p>
    <p>
        Then use browser full screen:
        <br>
        <b>Mac:</b> Control + Command + F
        <br>
        <b>Windows:</b> F11
    </p>
</div>
"""))


Closing server running on port: 7860


/tmp/ipykernel_1600/720473255.py:186: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(


Launching Gradio app...
In Colab, open the public gradio.live link in a new browser tab for the best full-screen view.
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://4b39f764eff15ec90e.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


#
---

# **How to Use the Gradio Interface**

### Steps

1. **Write your question** in the text box.

   Example:

   ```text
   What is the difference between artificial intelligence and machine learning?
   ```

2. **Choose the retrieval mode.**

   Recommended option:

   ```text
   hybrid_rrf
   ```

   Meaning of each option:

   - `bm25`: keyword-based search.
   - `semantic`: meaning-based search using embeddings.
   - `hybrid_rrf`: combines BM25 and semantic search using rank fusion.
   - `hybrid_score`: combines BM25 and semantic search using score fusion.

3. **Choose the number of retrieved sources.**

   Recommended value:

   ```text
   5
   ```

   Use more sources if the question is broad, and fewer sources if you want a shorter answer.

4. **Choose the generator.**

   Recommended option:

   ```text
   auto
   ```

   Meaning of each option:

   - `auto`: uses OpenAI if an API key exists, otherwise tries the local model.
   - `local`: uses the HuggingFace model.
   - `openai`: uses the OpenAI model if an API key is available.

5. **Set the temperature.**

   Recommended value:

   ```text
   0.1
   ```

   A low temperature gives more factual and stable answers.

6. Click **Generate answer**.

### What the Interface Shows

The interface will display:

- The generated answer.
- Citations such as `[S1]`, `[S2]`, and `[S3]`.
- A table of retrieved sources.

Each citation refers to a source in the table.

For example:

```text
[S1] = Source 1 in the retrieved sources table
[S2] = Source 2 in the retrieved sources table
[S3] = Source 3 in the retrieved sources table
```

### Best Demo Settings

Use these settings for the project demo:

```text
Retrieval mode: hybrid_rrf
Number of retrieved sources: 5
Generator: auto
Temperature: 0.1
```

### Good Demo Questions

```text
What is the difference between artificial intelligence and machine learning?
```

```text
How can neural networks handle varying input sizes?
```

```text
What is the difference between fit and fit_transform in scikit-learn?
```

### Note

The dataset mainly contains StackExchange question posts. Some answers may be limited because the corpus does not always include full accepted answers.

# **README.md**

## Smart Q&A Assistant with Hybrid Search

## Project Overview

This project builds a technical question-answering assistant using a hybrid search pipeline. The system retrieves relevant passages from a StackExchange-style corpus, combines keyword-based and semantic search, then generates an answer with citations from the retrieved sources.

The notebook is designed to run in Google Colab as a single self-contained cell.

## Dataset

The project uses two JSON files:

- `top_ai_questions.json`
- `top_datascience_questions.json`

Each file contains StackExchange-style question records with fields such as title, body, tags, score, view count, and source link.

## Main Features

The system includes:

1. Corpus loading and cleaning
2. HTML-to-text preprocessing
3. Passage chunking
4. BM25 lexical retrieval
5. MiniLM semantic retrieval
6. Hybrid retrieval using:
   - Reciprocal Rank Fusion
   - Score-based fusion
7. Retrieval evaluation using:
   - MAP@10
   - MRR@10
   - nDCG@10
8. RAG answer generation
9. Citation quality checking
10. Gradio user interface for live demo

## Retrieval Methods

### BM25

BM25 is used for keyword-based retrieval. It performs well when the query contains the same words or technical terms as the relevant document.

### Semantic Search

Semantic search uses the `sentence-transformers/all-MiniLM-L6-v2` embedding model. It retrieves passages based on meaning, even when the exact words are different.

### Hybrid Search

The system supports two hybrid methods:

- `hybrid_rrf`: combines BM25 and semantic rankings using Reciprocal Rank Fusion.
- `hybrid_score`: combines normalized BM25 and semantic scores.

The default mode is `hybrid_rrf`.

## RAG Generation

The RAG pipeline works as follows:

1. User asks a question.
2. The system retrieves top passages.
3. Each passage is labeled as `[S1]`, `[S2]`, etc.
4. A language model generates an answer using the retrieved context.
5. The answer includes citations from the retrieved sources.

The notebook supports:

- OpenAI generation if `OPENAI_API_KEY` is available.
- Local HuggingFace generation using `google/flan-t5-base` if no OpenAI key is available.
- Extractive fallback if generation fails.

## Gradio App

The notebook launches a Gradio interface where the user can:

- Enter a technical question
- Select retrieval mode
- Select number of retrieved sources
- Select generator type
- View the generated answer
- View retrieved citations and source links

## How to Run

1. Open Google Colab.
2. Upload:
   - `top_ai_questions.json`
   - `top_datascience_questions.json`
3. Paste the full notebook code into one Colab cell.
4. Run the cell.
5. Wait for package installation, indexing, evaluation, and Gradio launch.
6. Open the Gradio link printed at the end.

## Optional OpenAI Setup

To use OpenAI instead of the local HuggingFace model, run this before asking questions:

```python
import os
os.environ["OPENAI_API_KEY"] = "your_api_key_here"